# Day 10 — Loss Functions

## 1. Learning Objectives
- Understand what a Loss Function is fundamentally.
- Learn when to use **MSE** vs **MAE** for Regression.
- Learn when to use **BCE** vs **Cross Entropy** for Classification.
- Implement these using PyTorch's `nn` module.

## 2. Prerequisites
- Neural Network structure (`nn.Module`).

In [ ]:
import torch
import torch.nn as nn

## 3. Concept Explanation
A **Loss Function** (or Criterion/Cost Function) calculates how far off our model's predictions are from the true targets. 
- The goal of the optimizer is to minimize this number.
- The type of problem (Regression vs Classification) completely dictates which loss function you *must* use.

## 4. Regression Loss Functions
Use these when predicting continuous numbers (e.g., House Prices, Temperature).

1. **MSE (Mean Squared Error)**: `nn.MSELoss()`. Heavily penalizes large errors because it squares the difference. Good for standard regression.
2. **MAE (Mean Absolute Error)**: `nn.L1Loss()`. Doesn't square the error, so it is more robust to outliers in your dataset.

In [ ]:
# 8. Simple Example: Regression Loss
predictions = torch.tensor([100.0, 150.0, 200.0])
targets = torch.tensor([105.0, 145.0, 500.0]) # Notice the huge outlier (500)

mse = nn.MSELoss()
mae = nn.L1Loss()

print("MSE Loss:", mse(predictions, targets).item())
print("MAE Loss:", mae(predictions, targets).item())

## 5. Classification Loss Functions
Use these when predicting categories (e.g., Dog vs Cat, Spam vs Not Spam).

1. **BCE (Binary Cross Entropy)**: `nn.BCELoss()` or `nn.BCEWithLogitsLoss()`. Used when predicting exactly 2 classes (Output is a probability between 0 and 1).
2. **Cross Entropy**: `nn.CrossEntropyLoss()`. Used for multi-class classification (e.g., predicting digits 0-9). The model outputs raw scores (logits), and this function mathematically applies Softmax and calculates the loss.

In [ ]:
# Example: Multi-class Classification (Cross Entropy)
# Let's say we have 3 classes (Cat, Dog, Bird). Batch size of 2.
raw_logits = torch.tensor([ [2.0, 1.0, 0.1],  # Model strongly thinks sample 1 is Class 0 (Cat)
                            [0.5, 3.0, 0.2] ]) # Model strongly thinks sample 2 is Class 1 (Dog)

# True labels (Indices of the correct class)
targets = torch.tensor([0, 1]) # Correct labels: Cat, Dog

ce_loss = nn.CrossEntropyLoss()
loss = ce_loss(raw_logits, targets)

print("Cross Entropy Loss:", loss.item())

## 9. Code Walkthrough: BCE vs BCEWithLogits
A very common point of confusion:
- `nn.BCELoss()` expects probabilities (numbers strictly between 0 and 1). You must use a Sigmoid activation on your output layer.
- `nn.BCEWithLogitsLoss()` expects raw numbers (logits). It applies the Sigmoid internally. **This is preferred** as it is mathematically more stable.

## 11. Practice Exercise 1: Pick the right loss
For the following scenarios, state which PyTorch loss function you would use:
1. Predicting the age of a person from a photo.
2. Predicting whether an email is Spam or Not Spam.
3. Classifying an image into one of 1000 object categories.

**Solutions:**
1. `nn.MSELoss()` or `nn.L1Loss()` (Regression)
2. `nn.BCEWithLogitsLoss()` (Binary Classification)
3. `nn.CrossEntropyLoss()` (Multi-class Classification)

## 13. Debugging Challenge
Why does the following classification code crash?

In [ ]:
logits = torch.tensor([[1.5, 2.3, 0.1]]) # 1 sample, 3 classes

# We one-hot encoded our target like you would in Scikit-Learn (Class 1 is correct)
target = torch.tensor([[0.0, 1.0, 0.0]]) 

criterion = nn.CrossEntropyLoss()
# loss = criterion(logits, target) # Uncomment to see error

**Solution:** PyTorch's `nn.CrossEntropyLoss` (unlike Keras) does not expect one-hot encoded targets by default. It expects a **1D tensor of class indices**. 
The target should simply be `torch.tensor([1])`.

## 17. Interview Questions
1. **Why do we prefer `BCEWithLogitsLoss` over `BCELoss`?**
   *Answer*: Numerical stability. Computing Sigmoid and then Cross Entropy separately can cause floating-point underflow/overflow if values are extreme. `BCEWithLogitsLoss` combines them into one stable mathematical formula.
2. **If your dataset has many extreme outliers (e.g., housing prices with a few $100M mansions), should you use MSE or MAE?**
   *Answer*: MAE (`L1Loss`). MSE squares the error, meaning a single $100M error will massively skew the gradient and disrupt training. MAE handles outliers gracefully.

## 19. Day Summary
- Regression = MSE (`nn.MSELoss`) or MAE (`nn.L1Loss`).
- Binary Classification = `nn.BCEWithLogitsLoss` (Outputs 1 number, no Sigmoid needed).
- Multi-class Classification = `nn.CrossEntropyLoss` (Outputs N numbers, targets are class indices, no Softmax needed).